# Master Data Summary Statistics

Builds a presentation-ready slide deck with summary statistics (mean, std, min, max) for a curated set of game-level variables. Pulls per-team batting splits from `master_data.csv` and weather/start-hour columns from `league_weather_2021_2025.csv`, joined on `game_pk`.

**Output:** `master_data_summary.pptx` — black/white theme with `#DEFF9A` accents.

In [1]:
# If python-pptx is not installed, uncomment the next line:
# !pip install python-pptx

import pandas as pd
import numpy as np
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN

# Theme colors
WHITE = RGBColor(0xFF, 0xFF, 0xFF)
BLACK = RGBColor(0x00, 0x00, 0x00)
ACCENT = RGBColor(0xDE, 0xFF, 0x9A)
GRAY = RGBColor(0x88, 0x88, 0x88)

/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/avabrown/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
# ============================================================
# Load + merge master_data and league_weather on game_pk
# ============================================================

master = pd.read_csv('../../data/master_data.csv')
weather = pd.read_csv('../../data/league_weather_2021_2025.csv')

# Pull only what we need from each, then inner-join on game_pk
master_cols = ['game_pk', 'away_runs_scored', 'away_bat_hr', 'away_bat_k', 'away_bat_bb']
weather_cols = ['game_pk', 'start_hour', 'temp_f', 'rhum', 'pres', 'prcp',
                'wspd_mph', 'wind_cf', 'wind_lcf', 'wind_rcf']

merged = master[master_cols].merge(weather[weather_cols], on='game_pk', how='inner')

# Variable order + display labels
display_map = {
    'start_hour':       'Game Start Hour',
    'away_bat_hr':      'Away Home Runs',
    'away_runs_scored': 'Away Runs Scored',
    'away_bat_k':       'Away Strikeouts',
    'away_bat_bb':      'Away Walks',
    'temp_f':           'Temperature (°F)',
    'rhum':             'Humidity (%)',
    'pres':             'Pressure (hPa)',
    'prcp':             'Precipitation (mm)',
    'wspd_mph':         'Wind Speed (mph)',
    'wind_cf':          'Wind to CF',
    'wind_lcf':         'Wind to LCF',
    'wind_rcf':         'Wind to RCF',
}

ordered_cols = list(display_map.keys())
df = merged[ordered_cols].rename(columns=display_map)

print(f"Merged rows: {len(df):,}")
print(f"Variables:   {len(df.columns)}")
df.head()

Merged rows: 12,058
Variables:   13


,Game Start Hour,Away Home Runs,Away Runs Scored,Away Strikeouts,Away Walks,Temperature (°F),Humidity (%),Pressure (hPa),Precipitation (mm),Wind Speed (mph),Wind to CF,Wind to LCF,Wind to RCF
0,14.0,0.0,5,6.0,8.0,65.1,15.7,845.8,0.0,5.2,7.41,8.29,5.63
1,13.0,4.0,7,12.0,1.0,78.3,13.0,1010.4,0.0,10.2,0.95,-4.71,6.49
2,15.0,1.0,2,10.0,2.0,50.1,36.7,1012.7,0.0,14.1,-10.80,-16.95,-3.34
3,19.0,4.0,7,7.0,3.0,43.9,82.0,1012.8,0.0,5.0,-7.98,-7.70,-7.29
4,15.0,0.0,10,16.0,9.0,48.1,38.3,1000.7,0.0,4.0,-4.08,-2.13,-5.53


In [3]:
# ============================================================
# Compute summary statistics: mean, std, min, max
# ============================================================

summary = df.agg(['mean', 'std', 'min', 'max']).T.round(2)
summary.columns = ['Mean', 'Std', 'Min', 'Max']
summary.index.name = 'Variable'
summary

,Mean,Std,Min,Max
Variable,,,,
Game Start Hour,16.61,2.60,3.00,20.00
Away Home Runs,1.16,1.16,0.00,9.00
Away Runs Scored,4.43,3.24,0.00,28.00
Away Strikeouts,8.86,2.94,0.00,21.00
Away Walks,3.03,1.96,0.00,17.00
Temperature (°F),71.69,12.36,20.50,112.30
Humidity (%),63.16,19.68,2.30,100.00
Pressure (hPa),996.19,31.37,824.10,1032.80
Precipitation (mm),0.40,1.69,0.00,46.90


In [4]:
# ============================================================
# Build slide deck (16:9, black/white with #DEFF9A accents)
# ============================================================

prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

blank_layout = prs.slide_layouts[6]


def set_slide_bg(slide, color):
    bg = slide.background
    fill = bg.fill
    fill.solid()
    fill.fore_color.rgb = color


def add_textbox(slide, left, top, width, height, text, *,
                font_size=18, bold=False, color=BLACK, align=PP_ALIGN.LEFT,
                font_name='Helvetica'):
    tb = slide.shapes.add_textbox(left, top, width, height)
    tf = tb.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.name = font_name
    run.font.size = Pt(font_size)
    run.font.bold = bold
    run.font.color.rgb = color
    return tb


def add_accent_bar(slide, left, top, width, height, color=ACCENT):
    shape = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, left, top, width, height)
    shape.fill.solid()
    shape.fill.fore_color.rgb = color
    shape.line.fill.background()
    return shape


# ---------- Slide 1: Title ----------
slide1 = prs.slides.add_slide(blank_layout)
set_slide_bg(slide1, WHITE)

add_textbox(slide1, Inches(0.8), Inches(2.6), Inches(11.7), Inches(1.2),
            "Master Data Summary Statistics",
            font_size=44, bold=True, color=BLACK)

add_accent_bar(slide1, Inches(0.85), Inches(3.85), Inches(2.0), Inches(0.12))

add_textbox(slide1, Inches(0.8), Inches(4.05), Inches(11.7), Inches(0.6),
            "MLB Game-Level Metrics, 2021–2025",
            font_size=22, bold=False, color=BLACK)

add_textbox(slide1, Inches(0.8), Inches(6.7), Inches(11.7), Inches(0.4),
            f"n = {len(df):,} games",
            font_size=12, color=GRAY)


# ---------- Slide 2: Summary table ----------
slide2 = prs.slides.add_slide(blank_layout)
set_slide_bg(slide2, WHITE)

# Title
add_textbox(slide2, Inches(0.6), Inches(0.35), Inches(12.1), Inches(0.7),
            "Summary Statistics",
            font_size=28, bold=True, color=BLACK)

add_accent_bar(slide2, Inches(0.65), Inches(1.05), Inches(1.5), Inches(0.08))

# Build the table
n_rows = len(summary) + 1  # 1 header + 13 data
n_cols = 5  # Variable, Mean, Std, Min, Max

tbl_left = Inches(0.6)
tbl_top = Inches(1.35)
tbl_width = Inches(12.1)
tbl_height = Inches(5.6)

table_shape = slide2.shapes.add_table(n_rows, n_cols, tbl_left, tbl_top,
                                       tbl_width, tbl_height)
table = table_shape.table

# Column widths: Variable wider, stat columns equal
table.columns[0].width = Inches(4.5)
for c in range(1, 5):
    table.columns[c].width = Inches((12.1 - 4.5) / 4)

# Header row
headers = ['Variable', 'Mean', 'Std', 'Min', 'Max']
for c, h in enumerate(headers):
    cell = table.cell(0, c)
    cell.fill.solid()
    cell.fill.fore_color.rgb = ACCENT
    cell.text = ''
    tf = cell.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.LEFT if c == 0 else PP_ALIGN.RIGHT
    run = p.add_run()
    run.text = h
    run.font.name = 'Helvetica'
    run.font.size = Pt(14)
    run.font.bold = True
    run.font.color.rgb = BLACK

# Data rows
for r, (var, row) in enumerate(summary.iterrows(), start=1):
    values = [var,
              f"{row['Mean']:.2f}",
              f"{row['Std']:.2f}",
              f"{row['Min']:.2f}",
              f"{row['Max']:.2f}"]
    for c, val in enumerate(values):
        cell = table.cell(r, c)
        cell.fill.solid()
        cell.fill.fore_color.rgb = WHITE
        cell.text = ''
        tf = cell.text_frame
        tf.word_wrap = True
        p = tf.paragraphs[0]
        p.alignment = PP_ALIGN.LEFT if c == 0 else PP_ALIGN.RIGHT
        run = p.add_run()
        run.text = val
        run.font.name = 'Helvetica'
        run.font.size = Pt(12)
        run.font.color.rgb = BLACK

# Footer
add_textbox(slide2, Inches(0.6), Inches(7.05), Inches(12.1), Inches(0.35),
            f"n = {len(df):,} games  •  Source: master_data.csv + league_weather_2021_2025.csv",
            font_size=10, color=GRAY)


# Save
output_path = 'master_data_summary.pptx'
prs.save(output_path)
print(f"Saved: {output_path}")

Saved: master_data_summary.pptx
